In [32]:
import subprocess
import requests
from tqdm import tqdm
import os 
import pandas as pd
import numpy as np
from collections import Counter 


In [33]:
# start with cleaned dataframe from processing and cleaning steps
file_path = "/home/callum/projects/sno_analysis/data/processed"
df = pd.read_csv(f"{file_path}/df_clean.csv")

In [34]:
# Adding features

# Initialising charge and hydrophobicity lookups
# For charge, assumed neutral pH

KYTE_DOOLITTLE = {
    "A": 1.8, "R": -4.5, "N": -3.5, "D": -3.5, "C": 2.5,
    "Q": -3.5, "E": -3.5, "G": -0.4, "H": -3.2, "I": 4.5,
    "L": 3.8, "K": -3.9, "M": 1.9, "F": 2.8, "P": -1.6,
    "S": -0.8, "T": -0.7, "W": -0.9, "Y": -1.3, "V": 4.2
}


CHARGE = {
    "D": -1, "E": -1,   # acidic, negatively charged
    "K": 1, "R": 1,     # basic, positively charged
    "H": 0,             # often treated as neutral/weakly basic; some schemes give H = +1 — worth a footnote on which convention you picked
    # everything else defaults to 0
}





# define functions 

# add charges 
def get_charge_sum(seq, window = 5):
    
    cent = int(len(seq) // 2)    
    if seq[cent] == "C":  
        charges = []
        charges.append(0.0)
        start = -(window)
        end = window + 1
        for i in range(start, end):
            if i != 0:
                if seq[cent + i] in list(CHARGE.keys()):
                    charges.append(CHARGE[seq[cent + i]])
    
    return np.sum(charges)

def get_hydrophobicity_sum(seq, window = 5):
    
    cent = int(len(seq) // 2)    
    if seq[cent] == "C":  
        hphobics = []
        start = -(window)
        end = window + 1
        for i in range(start, end):
            if seq[cent + i] in ["-", "U"]:
                continue
            if i != 0:
                hphobics.append(KYTE_DOOLITTLE[seq[cent + i]])
    
    return np.sum(hphobics)

In [35]:
# call functions to add features 

df["hydrophobicity_5"] = df["sequence"].apply(lambda seq: get_hydrophobicity_sum(seq, window=5)
)  
df["hydrophobicity_10"] = df["sequence"].apply(lambda seq: get_hydrophobicity_sum(seq, window=10)
)  
df["charge_5"] = df["sequence"].apply(lambda seq: get_charge_sum(seq, window=5)
)  
df["charge_10"] = df["sequence"].apply(lambda seq: get_charge_sum(seq, window=10)
)  


In [36]:
# reducing df to just features and identifiers we are interested in 

pre_split = df[["id", "modified", "position", "sequence", "hydrophobicity_5", "hydrophobicity_10", "charge_5", "charge_10"]]

# save this to files
pre_split.to_csv(f"{file_path}/pre_split.csv")

In [37]:
# Next, we want to cluster together proteins that are similar to reduce redundancy and prevent data leakage when it comes to modelling

# set number of rows you want to work with - getting uniprot sequences and running CD-HIT can take a while
nrows = 24000

pre_split = pre_split.head(nrows)

# define function that gets sequences

def fetch_uniprot_sequences(ids, batch):
    missing = []
    id_dict = {}
    for i in tqdm(range(0, len(ids), batch)):
        
        chunk = ids[i : i + batch]
        #print(chunk)
        
        for uniprot_id in chunk:
            url = f"https://rest.uniprot.org/uniprotkb/{uniprot_id}.fasta"
            response = requests.get(url)
            
            #print(response.text)

            if response.status_code == 200:
                fasta = response.text
                #print(fasta)
                for line in fasta.splitlines():
                    if line.startswith(">"):
                        # f.write(f"\n")
                        parts = line.split("|")
                        
                        id = parts[1]
                        id_dict[id] = ""
                        #print(id)
                        #f.write(f">{id}\n")
                        continue
                        
                    # f.write(f"{line}\n")
                    id_dict[id] += line
            else:
                print(f"Failed to retrieve {uniprot_id}")
                missing.append(uniprot_id)
    return id_dict, missing


# set parameters before calling
ids = pre_split["id"]
batch = 80


# call function
dic, missing = fetch_uniprot_sequences(ids, batch)


 10%|█         | 31/300 [12:45<1:50:38, 24.68s/it]


SSLError: HTTPSConnectionPool(host='rest.uniprot.org', port=443): Max retries exceeded with url: /uniprotkb/O43865.fasta (Caused by SSLError(SSLZeroReturnError(6, 'TLS/SSL connection has been closed (EOF) (_ssl.c:1082)')))

In [ ]:
# def small function to convert sequences to fasta format

file_path = "/home/callum/projects/sno_analysis/data/processed/test"
out_path = "/home/callum/projects/sno_analysis/data/homology/int"
file_name = "Test.fasta"
def create_fasta(dic):
    with open(f"{file_path}/Test.fasta", "w") as f:
        for id, seq in dic.items():
            f.write(f">{id}\n{seq}\n")
            
    f.close()

# call fiunction
create_fasta(dic)

In [ ]:
# code for running cd-hit in terminal
code = "cd-hit -i /home/callum/projects/sno_analysis/data/processed/test/Test.fasta -o /home/callum/projects/sno_analysis/data/homology/int/test.txt -c 0.4 -n 2 -d 0"
code = code.split(" ")
subprocess.run(code) 

Program: CD-HIT, V4.8.1 (+OpenMP), Apr 24 2025, 22:00:32
Command: cd-hit -i
         /home/callum/projects/sno_analysis/data/processed/test/Test.fasta
         -o
         /home/callum/projects/sno_analysis/data/homology/int/test.txt
         -c 0.4 -n 2 -d 0

Started: Tue Oct  6 10:50:03 2026
                            Output                              
----------------------------------------------------------------
total seq: 20
longest and shortest : 2472 and 315
Total letters: 15721
Sequences have been sorted

Approximated minimal memory consumption:
Sequence        : 0M
Buffer          : 1 X 16M = 16M
Table           : 1 X 0M = 0M
Miscellaneous   : 0M
Total           : 16M

Table limit with the given memory limit:
Max number of representatives: 808948
Max number of word counting entries: 97930486

comparing sequences from          0  to         20

       20  finished         20  clusters

Approximated maximum memory consumption: 16M
writing new database
writing clustering inf

CompletedProcess(args=['cd-hit', '-i', '/home/callum/projects/sno_analysis/data/processed/test/Test.fasta', '-o', '/home/callum/projects/sno_analysis/data/homology/int/test.txt', '-c', '0.4', '-n', '2', '-d', '0'], returncode=0)

In [ ]:
# cd-hit returns a cluster file
# this needs to be parsed properly to get the cluster id for each protein

# got chat to write the rest to save time 

cluster_path = "/home/callum/projects/sno_analysis/data/homology/int/test.txt.clstr"

clusters = {}

with open(cluster_path) as f:
    for line in f:
        line = line.strip()

        if line.startswith(">Cluster"):
            cluster_id = line.split()[1]
            clusters[cluster_id] = []

        elif ", >" in line:
            protein_id = line.split(", >")[1].split("...")[0]
            clusters[cluster_id].append(protein_id)
            


# reverse dict

protein_to_cluster = {
    protein_id: cluster_id
    for cluster_id, protein_ids in clusters.items()
    for protein_id in protein_ids
}


pre_split["cluster_id"] = pre_split["id"].map(protein_to_cluster)



In [ ]:
# check current df looks correct
print(pre_split)

        id  modified  position                                   sequence  \
0   Q9Y2Z0         1        88  AKKSLELNPNNSTAMLRKGICEYHEKNYAAALETFTEGQKL   
1   Q14584         0       441  SNLSGHLRIHTGEKPFECLECGKAFTHSSSLNNHMRTHSAK   
2   P00390         0       328  SMVTAVPGRLPVMTMIPDVDCLLWAIGRVPNTKDLSLNKLG   
3   Q5UIP0         0      1735  TLECQHKRSRRVRRSKGCDCCGEKSQPQEKSLIGLKNTENN   
4   P0C8J8         0        17  ----MKTLIARHKAGEHIGICSVCSAHPLVIEAALAFDRNS   
5   Q9NYY3         1        26  TITYQPAASTKMCEQALGKGCGADSKKKRPPQPPEESQPPQ   
6   Q9Z1G4         0       730  AEEFDFGDTMVHQAIHTIEYCLGCISNTASYLRLWALSLAH   
7   Q9BXJ9         0       491  EGTSAVENLNEMQCMWFQTECAQAYKAMNKFGEALKKCHEI   
8   Q91VD9         0       128  KSKKAREGVMEFLLANHPLDCPICDQGGECDLQDQSMMFGS   
9   P07814         0       381  IQPHPRTGNKYNVYPTYDFACPIVDSIEGVTHALRTTEYHD   
10  P00533         0       506  QKTKIISNRGENSCKATGQVCHALCSPEGCWGPEPRDCVSC   
11  O55137         1        14  -------MEATLNLEPSGRSCWDEPLSIAVRGLAPEQPVTL   

In [ ]:
# next section aims to generate a position weight matrix
# first step is creating empty matrix with AAs as rows and position as columns

# positions will need to be set - depends on the sequence length in dataframe. currently, mine is 41

# amino acid letter
aas = ['A', 'R', 'N', 'D', 'C', 'E', 'Q', 'G', 'H', 'I', 'L', 'K', 'M', 'F', 'P', 'S', 'T', 'W', 'Y', 'V', 'U']


#determine how many columns to add - the sequence length
seq_length = len(pre_split["sequence"][0])

col_names = np.arange(1, seq_length + 1)

# Create empty matrix
matrix = pd.DataFrame(
    index=aas,
    columns=col_names
)
empty_matrix = matrix.fillna(0) 
print(empty_matrix)

  1  2  3  4  5  6  7  8  9  10  ... 32 33 34 35 36 37 38 39 40 41
A  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
R  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
N  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
D  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
C  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
E  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
Q  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
G  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
H  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
I  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
L  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
K  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
M  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  0  0
F  0  0  0  0  0  0  0  0  0  0  ...  0  0  0  0  0  0  0  0  

In [ ]:
# now populate the matrix before performing operations on it 

# get sequences as a series
seqs = pre_split["sequence"]

#define function
def clean_sequences(series):
    sequences = [seq for seq in series if "-" not in seq]  # drop windows with padding
    no_sequences = len(sequences)
    return sequences, no_sequences


# x is list of sequences ready for next step
x, y = clean_sequences(seqs)

In [ ]:


def get_pos_count(seq_list):
    pos_dict = {}
    for position in range(0, len(seq_list[0])):
        res_in_this_position = []
        for seq in seq_list:
            res_in_this_position.append(seq[position])
        pos_dict[position + 1] = Counter(res_in_this_position)
        
    return pos_dict

dic = get_pos_count(x)
print(dic)

# result is dict with count of each AA at each position in the sequence

{1: Counter({'A': 3, 'S': 2, 'T': 2, 'I': 2, 'Q': 2, 'L': 2, 'E': 1, 'K': 1, 'V': 1, 'N': 1}), 2: Counter({'K': 2, 'I': 2, 'T': 2, 'N': 1, 'M': 1, 'L': 1, 'E': 1, 'G': 1, 'S': 1, 'Q': 1, 'R': 1, 'C': 1, 'P': 1, 'V': 1}), 3: Counter({'T': 3, 'G': 3, 'A': 3, 'K': 2, 'L': 2, 'E': 2, 'V': 1, 'P': 1}), 4: Counter({'S': 3, 'T': 2, 'C': 2, 'Y': 2, 'K': 2, 'F': 1, 'H': 1, 'G': 1, 'D': 1, 'P': 1, 'Q': 1}), 5: Counter({'A': 4, 'Q': 3, 'G': 2, 'D': 2, 'P': 2, 'L': 1, 'I': 1, 'Y': 1, 'T': 1}), 6: Counter({'H': 2, 'V': 2, 'R': 2, 'I': 2, 'E': 1, 'P': 1, 'F': 1, 'Y': 1, 'N': 1, 'S': 1, 'L': 1, 'A': 1, 'T': 1}), 7: Counter({'E': 5, 'L': 2, 'A': 2, 'P': 1, 'K': 1, 'G': 1, 'T': 1, 'S': 1, 'V': 1, 'Q': 1, 'C': 1}), 8: Counter({'G': 4, 'N': 3, 'R': 2, 'A': 2, 'L': 2, 'D': 1, 'K': 1, 'S': 1, 'Y': 1}), 9: Counter({'T': 3, 'R': 2, 'S': 2, 'L': 2, 'W': 2, 'P': 1, 'I': 1, 'V': 1, 'N': 1, 'E': 1, 'A': 1}), 10: Counter({'N': 2, 'L': 2, 'R': 2, 'T': 2, 'M': 2, 'H': 1, 'K': 1, 'G': 1, 'A': 1, 'S': 1, 'E': 1, 'V':

In [ ]:
# next we populate the matrix we made with the dict we just generated 

def populate_matrix_with_dict(dict, matrix):

    for i, j in dict.items():
        for k in j.items():
            matrix.at[k[0] , i] = k[1]
    
    

    return matrix

matrix = populate_matrix_with_dict(dic, empty_matrix)

# convert nans to 0

print(matrix)

  1  2  3  4  5  6  7  8  9  10  ... 32 33 34 35 36 37 38 39 40 41
A  3  0  3  0  4  1  2  2  1  1  ...  0  3  1  0  2  2  0  0  3  1
R  0  1  0  0  0  2  0  2  2  2  ...  1  1  0  1  2  0  0  1  0  0
N  1  1  0  0  0  1  0  3  1  2  ...  1  1  0  0  0  1  1  0  2  1
D  0  0  0  1  2  0  0  1  0  0  ...  0  0  3  0  0  1  0  1  1  1
C  0  1  0  2  0  0  1  0  0  0  ...  0  1  0  0  1  0  2  0  0  1
E  1  1  2  0  0  1  5  0  1  1  ...  1  2  2  1  1  1  2  2  1  0
Q  2  1  0  1  3  0  1  0  0  0  ...  1  2  0  2  0  0  1  1  0  2
G  0  1  3  1  2  0  1  4  0  1  ...  4  1  1  2  0  2  2  0  2  3
H  0  0  0  1  0  2  0  0  0  1  ...  1  0  2  0  0  0  1  1  1  1
I  2  2  0  0  1  2  0  0  1  0  ...  0  1  0  1  0  0  2  0  1  1
L  2  1  2  0  1  1  2  2  2  2  ...  4  0  2  3  1  2  1  2  1  1
K  1  2  2  2  0  0  1  1  0  1  ...  0  2  0  0  4  1  1  1  1  2
M  0  1  0  0  0  0  0  0  0  2  ...  1  0  1  1  0  2  1  0  0  2
F  0  0  0  1  0  1  0  0  0  0  ...  1  0  0  1  0  0  0  1  

In [ ]:
# next is getting position probability matrix

def get_position_probability_matrix(matrix, no_sequences):
    matrix = matrix.to_numpy(dtype=float)   # force numeric dtype here
    
    matrix = matrix + 0.00001
    
    ppm = matrix/no_sequences
    
    return ppm

ppm = get_position_probability_matrix(matrix, y)

ppm = pd.DataFrame(ppm, index=matrix.index, columns=matrix.columns)



In [ ]:
# every column should sum to the number of sequences you fed in (y)
print(matrix.sum(axis=0))
assert (matrix.sum(axis=0) == y).all(), "column counts don't match sequence count — check indexing"

1     17
2     17
3     17
4     17
5     17
6     17
7     17
8     17
9     17
10    17
11    17
12    17
13    17
14    17
15    17
16    17
17    17
18    17
19    17
20    17
21    17
22    17
23    17
24    17
25    17
26    17
27    17
28    17
29    17
30    17
31    17
32    17
33    17
34    17
35    17
36    17
37    17
38    17
39    17
40    17
41    17
dtype: object


In [ ]:
print(ppm.sum(axis=0))  # should be ≈1 for every column

1     1.000012
2     1.000012
3     1.000012
4     1.000012
5     1.000012
6     1.000012
7     1.000012
8     1.000012
9     1.000012
10    1.000012
11    1.000012
12    1.000012
13    1.000012
14    1.000012
15    1.000012
16    1.000012
17    1.000012
18    1.000012
19    1.000012
20    1.000012
21    1.000012
22    1.000012
23    1.000012
24    1.000012
25    1.000012
26    1.000012
27    1.000012
28    1.000012
29    1.000012
30    1.000012
31    1.000012
32    1.000012
33    1.000012
34    1.000012
35    1.000012
36    1.000012
37    1.000012
38    1.000012
39    1.000012
40    1.000012
41    1.000012
dtype: float64


In [ ]:
# define different background values for each amino acid instead of using standard 1/20
background_freqs = {
    "A": 0.0825, "R": 0.0553, "N": 0.0406, "D": 0.0545, "C": 0.0137,
    "Q": 0.0393, "E": 0.0675, "G": 0.0707, "H": 0.0227, "I": 0.0596,
    "L": 0.0966, "K": 0.0584, "M": 0.0242, "F": 0.0386, "P": 0.0470,
    "S": 0.0656, "T": 0.0534, "W": 0.0108, "Y": 0.0292, "V": 0.0687,
    "U": 0.01
}

# convert to series
background_series = pd.Series(background_freqs).reindex(ppm.index)

# calculate pwm using row wise division
pwm = np.log2(ppm.div(background_series, axis=0))

In [ ]:

pwm = pd.DataFrame(pwm, index=matrix.index, columns=matrix.columns)
print(pwm)

          1          2          3          4          5          6   \
A   1.096967 -17.097641   1.096967 -17.097641   1.512003  -0.487986   
R -16.520527   0.089128 -16.520527 -16.520527 -16.520527   1.089121   
N   0.534928   0.534928 -16.074727 -16.074727 -16.074727   0.534928   
D -16.499503 -16.499503 -16.499503   0.110152   1.110144 -16.499503   
C -14.507423   2.102232 -14.507423   3.102225 -14.507423 -14.507423   
E  -0.198480  -0.198480   0.801513 -16.808135 -16.808135  -0.198480   
Q   1.581871   0.581878 -16.027776   0.581878   2.166831 -16.027776   
G -16.874957  -0.265302   1.319650  -0.265302   0.734690 -16.874957   
H -15.235939 -15.235939 -15.235939   1.373715 -15.235939   2.373708   
I   0.981088   0.981088 -16.628559 -16.628559  -0.018905   0.981088   
L   0.284377  -0.715615   0.284377 -17.325270  -0.715615  -0.715615   
K   0.010439   1.010432   1.010432   1.010432 -16.599215 -16.599215   
M -15.328254   1.281401 -15.328254 -15.328254 -15.328254 -15.328254   
F -16.

In [ ]:
def get_pwm_offset_scores(seq, pwm, offsets=(-3, -2, -1, 1, 2, 3)):
    cent = len(seq) // 2
    scores = {}
    for i in offsets:
        idx = cent + i
        residue = seq[idx]
        position = idx + 1
        scores[f"pwm_pos_{i:+d}"] = pwm.at[residue, position] if residue in pwm.index else np.nan
    return pd.Series(scores)

offset_df = pre_split["sequence"].apply(lambda seq: get_pwm_offset_scores(seq, pwm))
df_final = pd.concat([pre_split, offset_df], axis=1)

In [ ]:
print(df_final.shape)
df_final.to_csv(f"/home/callum/projects/sno_analysis/data/processed/full_df.csv")
print("ready for modelling!")

(20, 15)
ready for modelling!


In [ ]:
print(df_final.shape)




(20, 15)


In [ ]:
print(df_final.head())

       id  modified  position                                   sequence  \
0  Q9Y2Z0         1        88  AKKSLELNPNNSTAMLRKGICEYHEKNYAAALETFTEGQKL   
1  Q14584         0       441  SNLSGHLRIHTGEKPFECLECGKAFTHSSSLNNHMRTHSAK   
2  P00390         0       328  SMVTAVPGRLPVMTMIPDVDCLLWAIGRVPNTKDLSLNKLG   
3  Q5UIP0         0      1735  TLECQHKRSRRVRRSKGCDCCGEKSQPQEKSLIGLKNTENN   
4  P0C8J8         0        17  ----MKTLIARHKAGEHIGICSVCSAHPLVIEAALAFDRNS   

   hydrophobicity_5  hydrophobicity_10  charge_5  charge_10 cluster_id  \
0             -15.9              -16.6       1.0        1.0         18   
1               1.7              -10.2      -1.0       -1.0         10   
2              13.1               13.0      -2.0       -1.0         11   
3             -14.9              -38.3       0.0        3.0          0   
4               8.8                6.3      -1.0        1.0         14   

   pwm_pos_-3  pwm_pos_-2  pwm_pos_-1  pwm_pos_+1  pwm_pos_+2  pwm_pos_+3  
0    1.010432   -0.265